# Current Stress — Stacking Ensemble (Final)

Eksperimen ini dibuat agar proses pengembangan **Current Stress** terlihat iteratif dan terdokumentasi penuh ke MLflow (parameter, metrics, model, artifacts, dan output notebook).

In [ ]:
import os
os.environ["PYTHONWARNINGS"] = "ignore"

import warnings
warnings.simplefilter("ignore")

from pathlib import Path
import json
import tempfile

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.preprocessing import RobustScaler

try:
    from imblearn.over_sampling import SMOTE
    from imblearn.pipeline import Pipeline as ImbPipeline
    IMBLEARN_AVAILABLE = True
except Exception:
    IMBLEARN_AVAILABLE = False


In [ ]:
REPO_ROOT = Path.cwd().resolve().parents[2]
RAW_DATASET = REPO_ROOT / "Current-Stress" / "datasets" / "raw" / "student_lifestyle_dataset.csv"
EXPERIMENT_NAME = "Current-Stress-Experiments"

mlflow.set_experiment(EXPERIMENT_NAME)

df = pd.read_csv(RAW_DATASET)

def prepare_current_stress_dataset(frame: pd.DataFrame):
    data = frame.copy()
    mapping_stress = {'Low': 0, 'Moderate': 1, 'High': 2}
    mapping_performance = {'Poor': 0, 'Fair': 1, 'Good': 2, 'Excellent': 3}

    if 'Academic_Performance' in data.columns:
        data['Academic_Performance_Encoded'] = data['Academic_Performance'].map(mapping_performance)

    data['Stress_Level_Encoded'] = data['Stress_Level'].map(mapping_stress)

    drop_cols = [c for c in ['Student_ID', 'Stress_Level', 'Academic_Performance'] if c in data.columns]
    data = data.drop(columns=drop_cols)

    X = data.drop(columns=['Stress_Level_Encoded'])
    y = data['Stress_Level_Encoded']
    return data, X, y

prepared_df, X, y = prepare_current_stress_dataset(df)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=26,
    stratify=y
)

print('Prepared data shape:', prepared_df.shape)
print('Train/Test:', X_train.shape, X_test.shape)


In [ ]:
def _log_dataset_overview(dataframe: pd.DataFrame):
    mlflow.log_param('dataset_rows', int(dataframe.shape[0]))
    mlflow.log_param('dataset_columns', int(dataframe.shape[1]))
    mlflow.log_dict({
        'columns': dataframe.columns.tolist(),
        'dtypes': {k: str(v) for k, v in dataframe.dtypes.items()},
        'missing_values': dataframe.isna().sum().to_dict(),
    }, 'dataset/overview.json')


def _evaluate_multiclass(model, X_eval, y_eval):
    preds = model.predict(X_eval)
    metrics = {
        'accuracy': float(accuracy_score(y_eval, preds)),
        'f1_weighted': float(f1_score(y_eval, preds, average='weighted')),
        'precision_weighted': float(precision_score(y_eval, preds, average='weighted')),
        'recall_weighted': float(recall_score(y_eval, preds, average='weighted')),
    }

    if hasattr(model, 'predict_proba'):
        proba = model.predict_proba(X_eval)
        metrics['roc_auc_ovr'] = float(roc_auc_score(y_eval, proba, multi_class='ovr'))

    report = classification_report(y_eval, preds, output_dict=True)
    cm = confusion_matrix(y_eval, preds)
    return metrics, report, cm


def _log_eval_artifacts(report_dict, cm, artifact_prefix='evaluation'):
    mlflow.log_dict(report_dict, f'{artifact_prefix}/classification_report.json')

    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='PuBuGn', ax=ax)
    ax.set_title('Confusion Matrix')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    fig.tight_layout()

    with tempfile.TemporaryDirectory() as tmpdir:
        img = Path(tmpdir) / 'confusion_matrix.png'
        fig.savefig(img, dpi=140)
        mlflow.log_artifact(str(img), artifact_path=f'{artifact_prefix}/plots')
    plt.close(fig)


In [ ]:
with mlflow.start_run(run_name='Current Stress - Stacking Ensemble Final'):
    mlflow.set_tag('model_family', 'stacking_final')
    _log_dataset_overview(prepared_df)

    base_models = [
        ('lr', Pipeline([('scaler', RobustScaler()), ('clf', LogisticRegression(random_state=26, max_iter=1000))])),
        ('dt', DecisionTreeClassifier(random_state=26, max_depth=8)),
        ('rf', RandomForestClassifier(random_state=26, n_estimators=300, n_jobs=-1)),
    ]
    meta_model = LogisticRegression(random_state=26, max_iter=1000)

    stacking = StackingClassifier(
        estimators=base_models,
        final_estimator=meta_model,
        cv=5,
        passthrough=False,
        n_jobs=-1,
    )

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=26)
    cv_scores = cross_val_score(stacking, X_train, y_train, cv=cv, scoring='f1_weighted', n_jobs=-1)

    stacking.fit(X_train, y_train)
    metrics, report, cm = _evaluate_multiclass(stacking, X_test, y_test)

    mlflow.log_params({'model': 'StackingClassifier', 'cv_folds': 5, 'meta_model': 'LogisticRegression'})
    mlflow.log_metric('cv_f1_weighted_mean', float(cv_scores.mean()))
    mlflow.log_metric('cv_f1_weighted_std', float(cv_scores.std()))
    mlflow.log_metrics(metrics)
    _log_eval_artifacts(report, cm)

    signature = infer_signature(X_train.head(3), stacking.predict(X_train.head(3)))
    mlflow.sklearn.log_model(stacking, artifact_path='model', signature=signature, input_example=X_train.head(1))
